# 07 · Airflow, quality gate và RBAC

- DAG batch chạy đúng thứ tự upload → clean → daily → dims → yield → views → quality gate.
- Dữ liệu xấu bị chặn trước khi vào kho.
- `analyst` chỉ đọc được view; đọc bảng gốc bị từ chối.

In [1]:
import json
import subprocess
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ART = ROOT / "artifacts"
REPORTS = ROOT / "reports"

def sh(command, timeout=600):
    result = subprocess.run(command, capture_output=True, text=True, timeout=timeout)
    return result.stdout + result.stderr

def read_json(path):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() else {}

plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False})
GREEN, TEAL, GOLD, RED = "#1F6B45", "#2B6E7A", "#8F5E1A", "#A23B2A"

## 1. DAG đã được parse

In [2]:
print(sh(["docker", "exec", "airflow", "airflow", "dags", "list"], timeout=180))

dag_id             | fileloc                        | owners | is_paused
===================+================================+========+==========
agri_batch_dwh     | /opt/airflow/dags/agri_dags.py | agri   | False    
agri_ml_pipeline   | /opt/airflow/dags/agri_dags.py | agri   | False    
agri_retry_demo    | /opt/airflow/dags/agri_dags.py | agri   | False    
agri_stream_health | /opt/airflow/dags/agri_dags.py | agri   | False    
                                                                        



## 2. Chạy quality gate trên kho

In [3]:
print(sh(["docker", "exec", "agri-tools", "python", "/opt/agri/tests/quality_gate.py"], timeout=300)[-2500:])

ion.java:293', 'org.apache.hive.service.cli.operation.Operation:run:Operation.java:320', 'org.apache.hive.service.cli.session.HiveSessionImpl:executeStatementInternal:HiveSessionImpl.java:530', 'org.apache.hive.service.cli.session.HiveSessionImpl:executeStatement:HiveSessionImpl.java:506', 'sun.reflect.GeneratedMethodAccessor19:invoke::-1', 'sun.reflect.DelegatingMethodAccessorImpl:invoke:DelegatingMethodAccessorImpl.java:43', 'java.lang.reflect.Method:invoke:Method.java:498', 'org.apache.hive.service.cli.session.HiveSessionProxy:invoke:HiveSessionProxy.java:78', 'org.apache.hive.service.cli.session.HiveSessionProxy:access$000:HiveSessionProxy.java:36', 'org.apache.hive.service.cli.session.HiveSessionProxy$1:run:HiveSessionProxy.java:63', 'java.security.AccessController:doPrivileged:AccessController.java:-2', 'javax.security.auth.Subject:doAs:Subject.java:422', 'org.apache.hadoop.security.UserGroupInformation:doAs:UserGroupInformation.java:1746', 'org.apache.hive.service.cli.session.Hi

## 3. Dữ liệu xấu bị chặn

In [4]:
print(sh(["docker", "exec", "agri-tools", "python", "/opt/agri/tests/bad_batch_demo.py"], timeout=600)[-1800:])
display(read_json(ART / "quality" / "bad_batch_demo.json"))

Traceback (most recent call last):
  File "/opt/agri/tests/bad_batch_demo.py", line 75, in <module>
    raise SystemExit(main())
                     ^^^^^^
  File "/opt/agri/tests/bad_batch_demo.py", line 39, in main
    execute("DROP TABLE IF EXISTS agri_raw.bad_weather_hourly")
  File "/opt/agri/tooling/common.py", line 88, in execute
    return cursor.fetchall()
           ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/pyhive/common.py", line 142, in fetchall
    return list(iter(self.fetchone, None))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/pyhive/common.py", line 111, in fetchone
    self._fetch_while(lambda: not self._data and self._state != self._STATE_FINISHED)
  File "/usr/local/lib/python3.12/site-packages/pyhive/common.py", line 51, in _fetch_while
    self._fetch_more()
  File "/usr/local/lib/python3.12/site-packages/pyhive/hive.py", line 496, in _fetch_more
    raise ProgrammingError("No result set")
pyh

{}

## 4. RBAC: analyst đọc view, không đọc bảng gốc

In [5]:
beeline = ["docker", "exec", "hiveserver2", "beeline", "-u",
           "jdbc:hive2://localhost:10000/;auth=noSasl"]
print("--- analyst × view ---")
print(sh(beeline + ["-n", "analyst", "-e", "SELECT * FROM agri_dwh.v_benchmark_like_hourly LIMIT 5;"], timeout=300)[-800:])
print("--- analyst × raw table (phải bị từ chối) ---")
print(sh(beeline + ["-n", "analyst", "-e", "SELECT * FROM agri_dwh.fact_weather_hourly LIMIT 1;"], timeout=180)[-800:])

--- analyst × view ---


f39-bfc2-4d160cb290c8/hive_2026-11-12_06-34-25_531_7964414133417856678-3/-local-10005/HashTable-Stage-5/MapJoin-mapfile261--.hashtable (15183 bytes)
2026-11-12 06:34:35	Dump the side-table for tag: 1 with group count: 32 into file: file:/tmp/hive/3d9859ec-363a-4f39-bfc2-4d160cb290c8/hive_2026-11-12_06-34-25_531_7964414133417856678-3/-local-10005/HashTable-Stage-5/MapJoin-mapfile271--.hashtable
2026-11-12 06:34:35	Uploaded 1 File to: file:/tmp/hive/3d9859ec-363a-4f39-bfc2-4d160cb290c8/hive_2026-11-12_06-34-25_531_7964414133417856678-3/-local-10005/HashTable-Stage-5/MapJoin-mapfile271--.hashtable (1185 bytes)
2026-11-12 06:34:35	End of local task; Time Taken: 1.91 sec.
5 rows selected (26.167 seconds)
Beeline version 2.3.2 by Apache Hive
Closing: 0: jdbc:hive2://localhost:10000/;auth=noSasl

--- analyst × raw table (phải bị từ chối) ---


r:file:/opt/hadoop-2.7.4/share/hadoop/common/lib/slf4j-log4j12-1.7.10.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://localhost:10000/;auth=noSasl
Connected to: Apache Hive (version 2.3.2)
Driver: Hive JDBC (version 2.3.2)
Transaction isolation: TRANSACTION_REPEATABLE_READ
Error: Error while compiling statement: FAILED: HiveAccessControlException Permission denied: Principal [name=analyst, type=USER] does not have following privileges for operation QUERY [[SELECT] on Object [type=TABLE_OR_VIEW, name=agri_dwh.fact_weather_hourly]] (state=42000,code=40000)
Closing: 0: jdbc:hive2://localhost:10000/;auth=noSasl



## 5. Audit log của HiveServer2

In [6]:
print(sh(["docker", "exec", "hiveserver2", "bash", "-c",
          "ls -lt /tmp/hive/operation_logs 2>/dev/null | head -10; echo ---; "
          "grep -h 'user=' /tmp/hive/operation_logs/* 2>/dev/null | tail -5"]))

total 0
---



## Nghiệm thu P3

- DAG xanh, có retry (`agri_retry_demo` cố tình lỗi lần đầu rồi thành công ở lần hai).
- Quality gate 12/12 PASS; batch xấu bị chặn, số dòng kho không đổi.
- Analyst SELECT view thành công; SELECT bảng gốc trả `HiveAccessControlException`.
- HiveServer2 ghi operation log cho từng truy vấn.